# Story maps from Python

A story map walks the reader through a map, one stop at a time. Each stop is a **segment**: a place on the map, a title, and some text to read while you are there.

This notebook tells the story of one month of earthquakes in five stops, then you play it on the map.

The map is saved to `story-python-api.jGIS`, which gives it a toolbar to play the story from. The loop empties it first, so you can run this notebook again. The map itself is shown at the end, once the story is built.

In [ ]:
from jupytergis import GISDocument, constant, field

doc = GISDocument("story-python-api.jGIS", latitude=25, longitude=-100, zoom=2)
await doc.ready()

for layer in doc.layers:
    doc.remove_layer(layer)

## 1. Add the layers

A stop only moves the map. What the reader sees comes from ordinary layers:

- a light basemap, with place names on top,
- every earthquake the US Geological Survey recorded from 8 September to 8 October 2017, in orange and sized by magnitude,
- the strong ones, magnitude 5 and above, drawn again in red on top.

In [ ]:
import json

doc.add_raster_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    name="Basemap",
)
doc.add_raster_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Reference/MapServer/tile/{z}/{y}/{x}",
    name="Place names",
)

quakes = doc.add_geojson_layer(
    path="data/eq.geojson",
    name="Earthquakes",
    symbology=[field("mag").encoding("radius"), constant("orange").encoding("fill")],
)

with open("data/eq.geojson") as f:
    every_quake = json.load(f)
doc.add_geojson_layer(
    data={
        "type": "FeatureCollection",
        "features": [q for q in every_quake["features"] if q["properties"]["mag"] >= 5],
    },
    name="Magnitude 5 and above",
    symbology=[field("mag").encoding("radius"), constant("red").encoding("fill")],
)

## 2. Add the stops

Each `add_story_segment` call adds one stop, in order:

- `center` is the `(longitude, latitude)` the map flies to, and `zoom` how close it gets.
- `name` is the title on screen, and `markdown` the text under it.

Everything is optional and has a default. Run `help(doc.add_story_segment)` to see them all.

The first two stops ask a question: why are most of the dots in the United States?

In [ ]:
doc.add_story_segment(
    name="6,107 earthquakes",
    center=(-100, 25),
    zoom=2,
    markdown=(
        "Each dot is an earthquake recorded between 8 September and 8 October 2017. "
        "Bigger dots are stronger quakes, and red ones are magnitude 5 or more.\n\n"
        "Most of the dots crowd into Alaska and California. "
        "Does the ground really shake most there?"
    ),
)

doc.add_story_segment(
    name="Alaska",
    center=(-152, 61),
    zoom=4,
    markdown=(
        "Alaska alone has more than a third of the dots, and 9 in 10 of them "
        "are below magnitude 2.5, usually too weak to feel.\n\n"
        "The US has dense networks of seismometers that record these small quakes. "
        "The global network that covers the rest of the world mostly catches bigger ones: "
        "only 41 of its 1,240 quakes are below 2.5. "
        "The map is busiest where the instruments are."
    ),
)

The third stop uses `layer_overrides` to change another layer while it is on screen. It hides the layer with every earthquake, so only the red ones are left. The layer comes back when the reader moves on.

In [ ]:
doc.add_story_segment(
    name="The strong ones",
    center=(-120, 5),
    zoom=2,
    markdown=(
        "Keep only the 136 quakes of magnitude 5 and above, "
        "and Alaska and California almost disappear.\n\n"
        "What is left draws a ring around the Pacific Ocean, the Ring of Fire, "
        "where the plates of the Earth's crust meet."
    ),
    layer_overrides=[{"targetLayer": quakes, "visible": False}],
)

The last two stops zoom in on the two strongest quakes of the month, both in Mexico.

In [ ]:
doc.add_story_segment(
    name="Chiapas",
    center=(-94.5, 15.5),
    zoom=6,
    markdown=(
        "The strongest quake of the month, magnitude 8.1, struck off the Pacific coast of southern Mexico.\n\n"
        "In the month that followed, 341 more quakes shook the area within 250 km, "
        "38 of them of magnitude 5 or more."
    ),
)

doc.add_story_segment(
    name="Puebla",
    center=(-98.8, 19),
    zoom=7,
    markdown=(
        "On 19 September, a magnitude 7.1 quake struck 118 km south-east of Mexico City, "
        "on the anniversary of the 1985 earthquake that devastated the city.\n\n"
        "Unlike Chiapas, no other quake of the month was recorded within 150 km of it. "
        "Its dot stands alone."
    ),
)

## 3. Name the story

The first stop created the story with a default title. `doc.story` gives it back, and setting an attribute saves it.

In [ ]:
doc.story.title = "A month of earthquakes"

[segment.name for segment in doc.story.segments]

## 4. Play it

Show the map, then in its toolbar click **Open Story Editor** (the scroll icon) and **Preview story**. Move between the stops with the arrows on the sides of the map.

In [ ]:
doc